# Notebook 04 — Topic Modeling + Correlation Analysis
**Owner:** Person 4

**Inputs:**
- `../data/processed/sentiment_weekly.csv` (from Notebook 03)
- `../data/prices/prices_clean.csv` (from Notebook 01)
- `../data/reddit/reddit_raw.csv` (for LDA text corpus)

**Goal:**
1. Run LDA topic modeling on Reddit posts — identify dominant topics during price spikes vs. stable periods
2. Correlate weekly sentiment with hardware prices (including lagged correlation)
3. Produce final visualizations for the report

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import stats
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import warnings
warnings.filterwarnings('ignore')

## 1. Load All Data

In [ ]:
df_sentiment = pd.read_csv("../data/processed/sentiment_weekly.csv")
df_prices    = pd.read_csv("../data/prices/prices_clean.csv", parse_dates=["date"])
df_reddit    = pd.read_csv("../data/reddit/reddit_raw.csv", parse_dates=["date"])

print("Sentiment weekly:", df_sentiment.shape)
print("Prices:", df_prices.shape)
print("Reddit posts:", df_reddit.shape)

## 2. Prepare Weekly Price Index
Average price per product category per week.

In [ ]:
df_prices["week"] = df_prices["date"].dt.to_period("W").astype(str)

price_weekly = (
    df_prices.groupby(["week", "category"])
    ["price_usd"].mean()
    .reset_index()
    .rename(columns={"price_usd": "avg_price"})
)

# Also compute an overall price index (all categories)
price_index = (
    df_prices.groupby("week")["price_usd"]
    .mean()
    .reset_index()
    .rename(columns={"price_usd": "price_index"})
)

price_weekly.head()

## 3. Merge Sentiment + Price

In [ ]:
sentiment_all = df_sentiment[df_sentiment["subreddit"] == "ALL"].copy()
sentiment_all["week"] = sentiment_all["week"].astype(str)

df_merged = sentiment_all.merge(price_index, on="week", how="inner")
df_merged["week_dt"] = pd.PeriodIndex(df_merged["week"], freq="W").to_timestamp()
df_merged = df_merged.sort_values("week_dt").reset_index(drop=True)

print(f"Merged weeks: {len(df_merged)}")
df_merged.head()

## 4. Dual-Axis Timeline: Price + Sentiment

In [ ]:
fig, ax1 = plt.subplots(figsize=(15, 6))

color_price = "#e05c5c"
color_sent  = "#4a90d9"

ax1.set_xlabel("Week")
ax1.set_ylabel("Avg Hardware Price ($)", color=color_price)
ax1.plot(df_merged["week_dt"], df_merged["price_index"],
         color=color_price, linewidth=2, label="Price Index")
ax1.tick_params(axis="y", labelcolor=color_price)

ax2 = ax1.twinx()
ax2.set_ylabel("Avg Sentiment Score", color=color_sent)
ax2.plot(df_merged["week_dt"], df_merged["avg_sentiment"],
         color=color_sent, linewidth=2, linestyle="--", label="Sentiment")
ax2.axhline(0, color="gray", linestyle=":", linewidth=0.8)
ax2.tick_params(axis="y", labelcolor=color_sent)

ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
fig.autofmt_xdate()

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

plt.title("Hardware Price Index vs. Reddit Sentiment (Weekly)")
plt.tight_layout()
plt.savefig("../data/processed/price_vs_sentiment.png", dpi=150)
plt.show()

## 5. Correlation Analysis
### 5a. Contemporaneous (same week)

In [ ]:
pearson_r, pearson_p = stats.pearsonr(df_merged["avg_sentiment"], df_merged["price_index"])
spearman_r, spearman_p = stats.spearmanr(df_merged["avg_sentiment"], df_merged["price_index"])

print("=== Contemporaneous Correlation ===")
print(f"Pearson  r={pearson_r:.3f}  p={pearson_p:.4f}")
print(f"Spearman r={spearman_r:.3f}  p={spearman_p:.4f}")

### 5b. Lagged Correlation — Does sentiment predict future price (or vice versa)?

In [ ]:
lag_results = []
max_lag = 8  # weeks

for lag in range(-max_lag, max_lag + 1):
    sent = df_merged["avg_sentiment"]
    price = df_merged["price_index"].shift(lag)  # positive lag = sentiment leads price
    combined = pd.concat([sent, price], axis=1).dropna()
    if len(combined) > 5:
        r, p = stats.pearsonr(combined.iloc[:, 0], combined.iloc[:, 1])
        lag_results.append({"lag_weeks": lag, "pearson_r": r, "p_value": p})

df_lag = pd.DataFrame(lag_results)

plt.figure(figsize=(12, 5))
colors = ["red" if p < 0.05 else "steelblue" for p in df_lag["p_value"]]
plt.bar(df_lag["lag_weeks"], df_lag["pearson_r"], color=colors)
plt.axhline(0, color="black", linewidth=0.8)
plt.xlabel("Lag (weeks) — positive = sentiment leads price")
plt.ylabel("Pearson r")
plt.title("Lagged Correlation: Sentiment vs. Price\n(red bars = p < 0.05)")
plt.tight_layout()
plt.savefig("../data/processed/lagged_correlation.png", dpi=150)
plt.show()

best_lag = df_lag.loc[df_lag["pearson_r"].abs().idxmax()]
print(f"\nStrongest correlation at lag={best_lag['lag_weeks']} weeks (r={best_lag['pearson_r']:.3f}, p={best_lag['p_value']:.4f})")

## 6. LDA Topic Modeling
### 6a. Fit LDA on all posts

In [ ]:
# Use cleaned text if available from Notebook 03 output, otherwise fall back to raw title+body
corpus_text = (df_reddit["title"].fillna('') + " " + df_reddit["body"].fillna('')).tolist()

vectorizer = CountVectorizer(
    max_df=0.90,
    min_df=3,
    stop_words='english',
    max_features=3000
)
dtm = vectorizer.fit_transform(corpus_text)
vocab = vectorizer.get_feature_names_out()

N_TOPICS = 6
lda = LatentDirichletAllocation(n_components=N_TOPICS, random_state=42, max_iter=20)
lda.fit(dtm)
print(f"LDA fitted — {N_TOPICS} topics over {dtm.shape[0]} documents")

In [ ]:
def print_top_words(model, feature_names, n_top=10):
    for idx, topic in enumerate(model.components_):
        top_words = [feature_names[i] for i in topic.argsort()[:-n_top - 1:-1]]
        print(f"Topic {idx}: {', '.join(top_words)}")

print_top_words(lda, vocab)

### 6b. Assign dominant topic per post + analyze topic distribution during price spikes

In [ ]:
doc_topics = lda.transform(dtm)
df_reddit["dominant_topic"] = doc_topics.argmax(axis=1)
df_reddit["week"] = df_reddit["date"].dt.to_period("W").astype(str)

# Identify price spike weeks (top 25% price weeks)
price_index_copy = price_index.copy()
spike_threshold = price_index_copy["price_index"].quantile(0.75)
spike_weeks = set(price_index_copy[price_index_copy["price_index"] >= spike_threshold]["week"])

df_reddit["is_spike_week"] = df_reddit["week"].isin(spike_weeks)

topic_dist = df_reddit.groupby(["is_spike_week", "dominant_topic"]).size().unstack(fill_value=0)
topic_dist_pct = topic_dist.div(topic_dist.sum(axis=1), axis=0)

topic_dist_pct.index = ["Stable weeks", "Spike weeks"]
topic_dist_pct.columns = [f"Topic {i}" for i in topic_dist_pct.columns]

topic_dist_pct.T.plot(kind="bar", figsize=(12, 5), colormap="tab10")
plt.title("Topic Distribution: Spike Weeks vs. Stable Weeks")
plt.ylabel("Share of Posts")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("../data/processed/topic_distribution.png", dpi=150)
plt.show()

## 7. Summary of Findings

In [ ]:
print("=" * 60)
print("FINDINGS SUMMARY")
print("=" * 60)
print(f"\nData coverage:")
print(f"  Weeks analyzed:    {len(df_merged)}")
print(f"  Reddit posts:      {len(df_reddit)}")
print(f"  Price data points: {len(df_prices)}")
print(f"\nContemporaneous correlation (sentiment vs. price):")
print(f"  Pearson  r = {pearson_r:.3f}  (p = {pearson_p:.4f})")
print(f"  Spearman r = {spearman_r:.3f}  (p = {spearman_p:.4f})")
print(f"\nBest lagged correlation:")
print(f"  Lag = {best_lag['lag_weeks']} weeks  r = {best_lag['pearson_r']:.3f}  p = {best_lag['p_value']:.4f}")
print(f"\nSaved outputs to ../data/processed/")